# Modelos base sobre el índice de aptitud solar

Este cuaderno corre los modelos base del curso sobre la base limpia del EDA (`dataset_eda_corregido.csv`), en dos tareas:

- Clasificación de `solar_aptittude_class` (Baja, Media, Alta).
- Regresión de `solar_aptitude`, el índice continuo.

Tiene dos partes. Primero, los modelos con hiperparámetros por defecto en cuatro escenarios (bloques espaciales o partición aleatoria, con o sin latitud y longitud),
para ver cuánto depende el resultado de la ubicación. Segundo, los mismos modelos con hyperparameter tuning por validación anidada, en el escenario principal.

## Pipeline

La partición va primero. Todo lo que se aprende de los datos se ajusta solo con el entrenamiento de cada pliegue.

![Pipeline](fig_pipeline_benchmark_mundial.png)

## 1. Diseño

Datos: 57,976 plantas y las 15 predictoras del EDA corregido.

Validación externa: `StratifiedGroupKFold` con 5 pliegues, donde cada grupo es un bloque de 5° × 5° (`spatial_block`). Un bloque entero cae en entrenamiento o en prueba,
nunca partido. Con partición aleatoria, plantas vecinas quedan a los dos lados y el resultado sale optimista (Roberts et al., 2017; Ploton et al., 2020).

Fuga de información: la partición se hace primero. La imputación por mediana de `log_dist_to_road` (63 vacíos), el escalado y la calibración de probabilidades van
dentro de un `Pipeline`, así que se ajustan solo con el entrenamiento de cada pliegue. En el hyperparameter tuning, la búsqueda usa solo el entrenamiento
del pliegue externo y lo parte otra vez por bloques; el pliegue externo de prueba no se toca hasta medir.

Métricas de clasificación:
- F1 macro: promedio del F1 de las tres clases. Es la métrica principal, porque la clase Baja es el 3 % y la exactitud premiaría ignorarla.
- Exactitud y exactitud balanceada (promedio del recall por clase).
- Precisión macro y recall macro.
- F1 de la clase Baja, la minoritaria.
- AUC ROC macro uno contra el resto: qué tan bien ordenan las probabilidades a cada clase frente a las otras, sin depender de un umbral.
- Kappa ponderado cuadrático: las clases tienen orden (Baja < Media < Alta), y confundir Baja con Alta debe costar más que confundir Media con Alta.

Métricas de regresión: R², RMSE (castiga más los errores grandes) y MAE (error absoluto medio, en las unidades del índice).

Escala: el SVM con kernel RBF crece entre cuadrático y cúbico con el número de filas, así que se entrena con una submuestra de 8,000 filas del entrenamiento de cada pliegue.

In [ ]:
import time, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.base import BaseEstimator, ClassifierMixin, RegressorMixin, clone
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, label_binarize
from sklearn.linear_model import LogisticRegression, Ridge, Lasso
from sklearn.naive_bayes import GaussianNB
from sklearn.neighbors import KNeighborsClassifier, KNeighborsRegressor
from sklearn.svm import LinearSVC, SVC, SVR, LinearSVR
from sklearn.calibration import CalibratedClassifierCV
from sklearn.dummy import DummyClassifier, DummyRegressor
from sklearn.model_selection import StratifiedGroupKFold, StratifiedKFold, GroupKFold, GridSearchCV, cross_validate
from sklearn.metrics import (make_scorer, f1_score, cohen_kappa_score, accuracy_score, balanced_accuracy_score, precision_score,
                             recall_score, roc_auc_score, roc_curve, r2_score, mean_absolute_error, mean_squared_error,
                             confusion_matrix, ConfusionMatrixDisplay)

warnings.filterwarnings("ignore")
SEED = 42

df = pd.read_csv("dataset_eda_corregido.csv")
X15 = ["latitude", "longitude", "elevation", "slope", "curvature", "aspect_sin", "aspect_cos", "is_flat",
       "wind_sin", "wind_cos", "log_dist_to_road", "ambient_temperature", "humidity", "wind_speed", "ghi"]
X13 = [c for c in X15 if c not in ("latitude", "longitude")]
CLASES = ["Baja", "Media", "Alta"]                       # orden natural del índice
y_clf = pd.Categorical(df.solar_aptittude_class, categories=CLASES).codes
y_reg = df.solar_aptitude.values
grupos = df.spatial_block.astype(str).values

print("Filas:", len(df), "| bloques:", df.spatial_block.nunique())
print("Clases:", dict(zip(CLASES, np.bincount(y_clf))))
print("Vacíos por columna:", df[X15].isna().sum()[lambda s: s > 0].to_dict())

In [ ]:
class Submuestra(BaseEstimator):
    # Entrena el estimador en una submuestra aleatoria del entrenamiento (SVM con kernel). Expone probabilidades si el estimador las tiene.
    def __init__(self, est=None, n=8000, random_state=SEED):
        self.est, self.n, self.random_state = est, n, random_state
    def fit(self, X, y):
        rng = np.random.default_rng(self.random_state)
        i = rng.choice(len(X), min(self.n, len(X)), replace=False)
        self.est_ = clone(self.est).fit(np.asarray(X)[i], np.asarray(y)[i])
        if hasattr(self.est_, "classes_"):
            self.classes_ = self.est_.classes_
        return self
    def predict(self, X): return self.est_.predict(X)
    def predict_proba(self, X): return self.est_.predict_proba(X)

class SubClf(ClassifierMixin, Submuestra): pass
class SubReg(RegressorMixin, Submuestra): pass

def pipe(modelo):
    return Pipeline([("imp", SimpleImputer(strategy="median")), ("esc", StandardScaler()), ("m", modelo)])

CLF = {
    "Referencia: clase mayoritaria": DummyClassifier(strategy="most_frequent"),
    "Regresión logística": LogisticRegression(max_iter=1000),
    "Regresión logística balanceada": LogisticRegression(max_iter=1000, class_weight="balanced"),
    "Naive Bayes gaussiano": GaussianNB(),
    "KNN": KNeighborsClassifier(n_neighbors=15),
    "SVM lineal (calibrado)": CalibratedClassifierCV(LinearSVC(dual=False), cv=3),
    "SVM RBF (8 mil filas)": SubClf(SVC(kernel="rbf", probability=True, random_state=SEED)),
}
REG = {
    "Referencia: media": DummyRegressor(),
    "Ridge": Ridge(alpha=1.0),
    "Lasso": Lasso(alpha=0.001, max_iter=10000),
    "KNN": KNeighborsRegressor(n_neighbors=15),
    "SVR lineal": LinearSVR(dual=False, loss="squared_epsilon_insensitive", random_state=SEED),
    "SVR RBF (8 mil filas)": SubReg(SVR(kernel="rbf")),
}

kappa_c = make_scorer(cohen_kappa_score, weights="quadratic")
f1_baja = make_scorer(f1_score, labels=[0], average="macro")
SC_CLF = {"f1_macro": "f1_macro", "exactitud": "accuracy", "exactitud_bal": "balanced_accuracy", "precision_macro": "precision_macro",
          "recall_macro": "recall_macro", "f1_baja": f1_baja, "auc_ovr": "roc_auc_ovr", "kappa_cuad": kappa_c}
SC_REG = {"r2": "r2", "rmse": "neg_root_mean_squared_error", "mae": "neg_mean_absolute_error"}

def particiones(tipo):
    if tipo == "bloques 5°":
        return list(StratifiedGroupKFold(5, shuffle=True, random_state=SEED).split(df, y_clf, grupos))
    return list(StratifiedKFold(5, shuffle=True, random_state=SEED).split(df, y_clf))

## 2. Modelos con hiperparámetros por defecto, cuatro escenarios

Cada celda es el promedio de los 5 outer folds. `_de` es la desviación estándar entre pliegues.

In [ ]:
def corre(tarea):
    modelos, y, sc = (CLF, y_clf, SC_CLF) if tarea == "clf" else (REG, y_reg, SC_REG)
    filas = []
    for vars_nombre, cols in [("con lat/lon", X15), ("sin lat/lon", X13)]:
        for tipo in ["bloques 5°", "aleatoria"]:
            cv = particiones(tipo)
            for nombre, modelo in modelos.items():
                t0 = time.time()
                r = cross_validate(pipe(modelo), df[cols].values, y, cv=cv, scoring=sc, n_jobs=-1, error_score=np.nan)
                fila = {"modelo": nombre, "variables": vars_nombre, "validación": tipo, "segundos": time.time() - t0}
                for k in sc:
                    v = r[f"test_{k}"]
                    v = -v if k in ("rmse", "mae") else v
                    fila[k], fila[k + "_de"] = np.nanmean(v), np.nanstd(v)
                filas.append(fila)
    return pd.DataFrame(filas)

res_clf = corre("clf")
res_clf.to_csv("resultados_benchmark_clasificacion.csv", index=False)
res_clf.pivot_table(index="modelo", columns=["variables", "validación"], values="f1_macro").reindex(list(CLF)).round(3)

In [ ]:
principal = res_clf[(res_clf.validación == "bloques 5°") & (res_clf.variables == "con lat/lon")].set_index("modelo").reindex(list(CLF))
principal[list(SC_CLF)].round(3)

In [ ]:
res_reg = corre("reg")
res_reg.to_csv("resultados_benchmark_regresion.csv", index=False)
display(res_reg.pivot_table(index="modelo", columns=["variables", "validación"], values="r2").reindex(list(REG)).round(3))
res_reg[(res_reg.validación == "bloques 5°") & (res_reg.variables == "con lat/lon")].set_index("modelo").reindex(list(REG))[["r2", "r2_de", "rmse", "mae", "segundos"]].round(4)

## 3. Dejar una región completa fuera

Se entrena con dos macrorregiones y se predice la tercera. Es la prueba más dura: mide si el modelo sirve en un continente que no vio. Regresión, R².

In [ ]:
region = df.macro_region.values
filas = []
for vars_nombre, cols in [("con lat/lon", X15), ("sin lat/lon", X13)]:
    for nombre, modelo in REG.items():
        for r in np.unique(region):
            te = region == r
            m = pipe(clone(modelo)).fit(df.loc[~te, cols].values, y_reg[~te])
            filas.append({"modelo": nombre, "variables": vars_nombre, "región fuera": r,
                          "r2": r2_score(y_reg[te], m.predict(df.loc[te, cols].values))})
lro = pd.DataFrame(filas)
lro.to_csv("resultados_benchmark_region_fuera.csv", index=False)
lro.pivot_table(index="modelo", columns=["variables", "región fuera"], values="r2").reindex(list(REG)).round(2)

Un R² negativo significa que el modelo predice peor que el promedio de la región de prueba. Hasta la referencia sale negativa, porque el promedio
del índice cambia de una región a otra.

## 4. hyperparameter tuning con validación anidada

Escenario principal: bloques de 5° y con latitud y longitud.

- Bucle externo: los mismos 5 pliegues por bloques. Solo mide.
- Bucle interno: dentro del entrenamiento de cada pliegue externo, `GridSearchCV` con 3 pliegues que también agrupan por bloque. Elige los hiperparámetros
  con F1 macro (clasificación) o R² (regresión) y reentrena con todo el entrenamiento externo.
- El `Pipeline` completo (imputar, escalar, calibrar, modelo) se ajusta dentro de cada pliegue interno, así que la búsqueda tampoco ve datos de validación.

Las grillas usan escala logarítmica para los parámetros de regularización (C, alpha), como recomienda el enunciado. Son pequeñas a propósito: el objetivo es un
benchmark base honesto, no exprimir cada modelo. Se reporta el resultado externo y los hiperparámetros elegidos en cada pliegue, para ver si son estables.

In [ ]:
GRID_CLF = {
    "Regresión logística": {"m__C": [0.01, 0.1, 1, 10, 100]},
    "Regresión logística balanceada": {"m__C": [0.01, 0.1, 1, 10, 100]},
    "Naive Bayes gaussiano": {"m__var_smoothing": [1e-9, 1e-7, 1e-5, 1e-3]},
    "KNN": {"m__n_neighbors": [5, 15, 31, 51], "m__weights": ["uniform", "distance"]},
    "SVM lineal (calibrado)": {"m__estimator__C": [0.01, 0.1, 1, 10]},
    "SVM RBF (8 mil filas)": {"m__est__C": [1, 10], "m__est__gamma": ["scale", 0.3]},
}
GRID_REG = {
    "Ridge": {"m__alpha": [0.001, 0.01, 0.1, 1, 10, 100, 1000]},
    "Lasso": {"m__alpha": [1e-5, 1e-4, 1e-3, 1e-2]},
    "KNN": {"m__n_neighbors": [5, 15, 31, 51], "m__weights": ["uniform", "distance"]},
    "SVR lineal": {"m__C": [0.01, 0.1, 1, 10]},
    "SVR RBF (8 mil filas)": {"m__est__C": [1, 10], "m__est__epsilon": [0.01, 0.05]},
}

def metricas_clf(y, pred, proba):
    return {"f1_macro": f1_score(y, pred, average="macro"), "exactitud": accuracy_score(y, pred),
            "exactitud_bal": balanced_accuracy_score(y, pred), "precision_macro": precision_score(y, pred, average="macro", zero_division=0),
            "recall_macro": recall_score(y, pred, average="macro"), "f1_baja": f1_score(y, pred, labels=[0], average="macro"),
            "auc_ovr": roc_auc_score(y, proba, multi_class="ovr", average="macro", labels=[0, 1, 2]),
            "kappa_cuad": cohen_kappa_score(y, pred, weights="quadratic")}

def metricas_reg(y, pred):
    return {"r2": r2_score(y, pred), "rmse": np.sqrt(mean_squared_error(y, pred)), "mae": mean_absolute_error(y, pred)}

def anidada(tarea, cols=X15):
    modelos, grids, y = (CLF, GRID_CLF, y_clf) if tarea == "clf" else (REG, GRID_REG, y_reg)
    X = df[cols].values
    externo = particiones("bloques 5°")
    filas, params, probas = [], [], {}
    for nombre, grid in grids.items():
        t0 = time.time()
        oof = np.zeros((len(df), 3)) if tarea == "clf" else None
        for k, (tr, te) in enumerate(externo):
            interno = StratifiedGroupKFold(3, shuffle=True, random_state=SEED) if tarea == "clf" else GroupKFold(3)
            busq = GridSearchCV(pipe(clone(modelos[nombre])), grid, cv=interno,
                                scoring="f1_macro" if tarea == "clf" else "r2", n_jobs=-1, refit=True)
            busq.fit(X[tr], y[tr], groups=grupos[tr])
            if tarea == "clf":
                proba = busq.predict_proba(X[te]); oof[te] = proba
                m = metricas_clf(y[te], busq.predict(X[te]), proba)
            else:
                m = metricas_reg(y[te], busq.predict(X[te]))
            filas.append({"modelo": nombre, "pliegue": k, **m})
            params.append({"modelo": nombre, "pliegue": k, **{p.replace("m__", ""): v for p, v in busq.best_params_.items()}})
        if tarea == "clf":
            probas[nombre] = oof
        print(f"{nombre}: {time.time() - t0:.0f} s")
    return pd.DataFrame(filas), pd.DataFrame(params), probas

anid_clf, par_clf, probas_clf = anidada("clf")
anid_clf.to_csv("resultados_anidada_clasificacion.csv", index=False)
tab_anid_clf = anid_clf.groupby("modelo").mean(numeric_only=True).drop(columns="pliegue").reindex(list(GRID_CLF))
tab_anid_clf.round(3)

In [ ]:
par_clf.set_index(["modelo", "pliegue"])

In [ ]:
comp_clf = pd.DataFrame({"F1 macro por defecto": principal.f1_macro, "F1 macro ajustado": tab_anid_clf.f1_macro,
                         "AUC por defecto": principal.auc_ovr, "AUC ajustado": tab_anid_clf.auc_ovr}).reindex(list(GRID_CLF))
comp_clf.round(3)

In [ ]:
anid_reg, par_reg, _ = anidada("reg")
anid_reg.to_csv("resultados_anidada_regresion.csv", index=False)
tab_anid_reg = anid_reg.groupby("modelo").mean(numeric_only=True).drop(columns="pliegue").reindex(list(GRID_REG))
base_reg = res_reg[(res_reg.validación == "bloques 5°") & (res_reg.variables == "con lat/lon")].set_index("modelo")
display(tab_anid_reg.round(4))
display(par_reg.set_index(["modelo", "pliegue"]))
pd.DataFrame({"R² por defecto": base_reg.r2, "R² ajustado": tab_anid_reg.r2}).reindex(list(GRID_REG)).round(3)

## 5. Gráficas

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 5.5))
for ax, (res, met, orden, titulo) in zip(axes, [(res_clf, "f1_macro", list(CLF), "Clasificación, F1 macro"), (res_reg, "r2", list(REG), "Regresión, R²")]):
    t = res.pivot_table(index="modelo", columns=["variables", "validación"], values=met).reindex(orden)
    t = t.drop(index=[i for i in t.index if i.startswith("Referencia")])
    xs = np.arange(len(t.index)); w = 0.2
    for k, col in enumerate([("con lat/lon", "aleatoria"), ("con lat/lon", "bloques 5°"), ("sin lat/lon", "aleatoria"), ("sin lat/lon", "bloques 5°")]):
        ax.bar(xs + (k - 1.5) * w, t[col], w, label=f"{col[0]}, {col[1]}")
    ax.set_xticks(xs); ax.set_xticklabels(t.index, rotation=25, ha="right", fontsize=9)
    ax.set_title(titulo); ax.axhline(0, color="k", lw=.6)
axes[0].legend(fontsize=8)
fig.suptitle("Hiperparámetros por defecto: cuánto depende el resultado de la ubicación y de la partición", y=1.02)
fig.tight_layout(); fig.savefig("fig_benchmark_modelos.png", dpi=130, bbox_inches="tight"); plt.show()

In [ ]:
mejor = tab_anid_clf.f1_macro.idxmax()
proba = probas_clf[mejor]
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
Y = label_binarize(y_clf, classes=[0, 1, 2])
for c, nombre in enumerate(CLASES):
    fpr, tpr, _ = roc_curve(Y[:, c], proba[:, c])
    axes[0].plot(fpr, tpr, label=f"{nombre} (AUC {roc_auc_score(Y[:, c], proba[:, c]):.3f})")
axes[0].plot([0, 1], [0, 1], "k--", lw=.8); axes[0].set_xlabel("tasa de falsos positivos"); axes[0].set_ylabel("tasa de verdaderos positivos")
axes[0].set_title(f"Curvas ROC uno contra el resto\n{mejor}, ajustado, bloques 5°"); axes[0].legend()
ConfusionMatrixDisplay(confusion_matrix(y_clf, proba.argmax(1), normalize="true"), display_labels=CLASES).plot(ax=axes[1], values_format=".2f", colorbar=False)
axes[1].set_title("Matriz de confusión (filas = clase real, normalizado)")
fig.tight_layout(); fig.savefig("fig_benchmark_roc_confusion.png", dpi=130, bbox_inches="tight"); plt.show()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 5))
comp_clf[["F1 macro por defecto", "F1 macro ajustado"]].plot.barh(ax=axes[0]); axes[0].set_title("Clasificación: efecto del ajuste (bloques 5°)")
pd.DataFrame({"R² por defecto": base_reg.r2, "R² ajustado": tab_anid_reg.r2}).reindex(list(GRID_REG)).plot.barh(ax=axes[1])
axes[1].set_title("Regresión: efecto del ajuste (bloques 5°)")
fig.tight_layout(); fig.savefig("fig_benchmark_ajuste.png", dpi=130, bbox_inches="tight"); plt.show()

## 6. Cómo leer los resultados

- Si un modelo cae mucho al quitar latitud y longitud, una parte grande de lo que aprende es la ubicación, no el terreno.
- Si cae al pasar de partición aleatoria a bloques, se estaba apoyando en plantas vecinas.
- Si dejando una región fuera sale negativo, no sirve fuera de los continentes que vio.
- Si el ajuste mejora poco, el límite está en los datos y no en los hiperparámetros.

## 8. Conclusiones

Con hiperparámetros ajustados por validación anidada, en bloques de 5° y con latitud y longitud:
- Clasificación: el SVM RBF es el mejor (F1 macro 0.818, AUC 0.960, kappa ponderado 0.775), seguido de KNN (F1 0.797, AUC 0.925). La regresión logística balanceada
  tiene la mejor exactitud balanceada (0.798) pero un F1 macro de 0.629. El SVM lineal (0.516) y Naive Bayes (0.472) quedan abajo. La referencia da 0.285.
- Regresión: el SVR RBF (R² 0.806) y KNN (0.796) superan con claridad a Ridge, Lasso y el SVR lineal (0.479). La referencia da -0.002.

Los contrastes con hiperparámetros por defecto muestran que esos resultados dependen de la ubicación. Sin latitud y longitud, KNN baja de 0.79 a 0.48 en R².
Con partición aleatoria los resultados salen más altos que con bloques. Dejando una región completa fuera, todos los modelos dan R² negativo, incluida la referencia.
Los modelos lineales quedan bajos porque el índice tiene un nivel distinto en cada región, y un solo plano no puede representarlo. KNN y los SVM con kernel
ajustan vecindarios locales y aprenden ese nivel.

El ajuste ayudó sobre todo al SVR RBF (de 0.734 a 0.806), porque el margen `epsilon` por defecto de 0.1 era grande para un índice que va de 0 a 0.94. A los modelos lineales casi no les cambió nada.

Límites: grillas pequeñas, SVM con kernel entrenados con 8,000 filas, sin árboles ni balanceo con SMOTE o ADASYN, que quedan para la entrega siguiente. La interpretación
completa y las propuestas de mejora con literatura están en el Resumen final.

# Resumen final

Este apartado es el mismo en los cuatro cuadernos del proyecto. Resume todo el trabajo, desde la recolección de los datos hasta los benchmarks, con las métricas,
las decisiones y la razón de cada una. Las cifras salen de la ejecución de los cuadernos el 2 de octubre de 2026.

| Cuaderno | Qué hace |
|---|---|
| `EDA_Corregido(Entregable 3).ipynb` | Limpia y explora la base mundial y produce `dataset_eda_corregido.csv` |
| `Benchmark_Modelos_Base.ipynb` | Modelos base del curso sobre la base mundial |
| `Clima_a_Generacion_Colombia.ipynb` | Construye la base de generación real de Colombia, la explora y prueba modelos preliminares |
| `Benchmark_Colombia.ipynb` | Modelos base del curso sobre la base de Colombia |

## R0. Guion de la presentación, diapositiva por diapositiva

Cada diapositiva dice qué mostrar, qué decir y el número que hay que citar. El detalle y las referencias de cada punto están en las secciones R1 a R15 de este resumen.

**Diapositiva 1. Título y pregunta.**
Qué decir: queríamos predecir qué tan apto es un lugar para una planta solar, a partir del clima, para que sirviera en cualquier parte. Trabajamos con un dataset
mundial publicado en 2026 (Mantilla-Guerra et al., 2026) de 58,978 plantas solares.

**Diapositiva 2. Los datos.**
Mostrar: la tabla de fuentes de R2.
Qué decir: el dataset une cinco fuentes. Global Energy Monitor da las plantas, Global Solar Atlas el potencial solar, ERA5 y NASA POWER el clima, los modelos
de elevación el terreno y OpenStreetMap las carreteras. Son 29 columnas. La variable a predecir es un índice de aptitud solar de 0 a 1 y su versión en tres clases:
Baja, Media y Alta.

**Diapositiva 3. Qué mide de verdad el índice (el primer hallazgo).**
Mostrar: la ecuación del índice de R1.
Qué decir: el paper calcula el índice solo con pendiente, orientación, sombreado y curvatura. El clima no entra. Por eso, aunque quisiéramos, el clima no puede
predecir este índice. Esto cambió el proyecto.

**Diapositiva 4. Pipeline del EDA.**
Mostrar: `fig_pipeline_eda_mundial.png`.
Qué decir: recorrer las cajas. De 58,978 plantas a 57,976 tras quitar filas rotas y duplicados. Se crean variables nuevas, se eligen 15 predictoras y se asigna
cada planta a un bloque del mapa de 5° × 5°.

**Diapositiva 5. Problemas de calidad de datos.**
Mostrar: la tabla de problemas de R3.
Qué decir: 64 plantas sin datos de terreno, todas en la clase Baja. 1,598 duplicados. Distancias a carretera de hasta 2,296 km, que son imposibles. Un 27.7 % de
áreas vacías. Para cada una, la tabla dice qué decisión se tomó y por qué. Los ángulos se pasan a seno y coseno porque 359° y 1° son vecinos.

**Diapositiva 6. El hallazgo central del EDA.**
Mostrar: `fig_mapa_clases_e3.png` o `fig_correlacion_e3.png` del EDA.
Qué decir:
- El índice casi no se relaciona con la pendiente (Spearman -0.07), con la que supuestamente se calcula, y sí con la longitud (+0.58).
- Con solo terreno, un modelo explica R² de 0.18. Con latitud y longitud, 0.91.
- El 94.5 % de la clase Baja viene de un lote de plantas europeas añadido después.

Conclusión: el índice mide sobre todo la región y el lote de procesamiento, no el terreno.

**Diapositiva 7. Por qué validación espacial.**
Mostrar: `fig_pipeline_benchmark_mundial.png`.
Qué decir: si partimos los datos al azar, plantas vecinas quedan una en entrenamiento y otra en prueba, y el modelo "hace trampa" por vecindad (Roberts et al., 2017).
Por eso partimos por bloques de 5° × 5°: un bloque entero va a entrenamiento o a prueba. Además, todo lo que se aprende de los datos (la mediana para rellenar
vacíos, la escala) se ajusta solo con el entrenamiento, dentro de un `Pipeline`, para no filtrar información de la prueba (Kaufman et al., 2012).

**Diapositiva 8. Modelos y métricas.**
Mostrar: las tablas de R6 y R7.
Qué decir: usamos los modelos base del curso. En clasificación, regresión logística, Naive Bayes, KNN y SVM. En regresión, Ridge, Lasso, KNN y SVR. La métrica principal
de clasificación es F1 macro, porque la clase Baja es solo el 3 % y la exactitud premiaría ignorarla. También reportamos AUC, que mide si las probabilidades
ordenan bien sin depender de un umbral, y kappa ponderado, porque las clases tienen orden. En regresión, R², RMSE y MAE.

**Diapositiva 9. hyperparameter tuning sin trampa.**
Mostrar: el esquema de validación anidada de R5.
Qué decir: si elegimos los hiperparámetros mirando los mismos datos con los que reportamos el resultado, el resultado sale inflado (Varma y Simon, 2006).
Por eso usamos validación anidada: un bucle interno elige los hiperparámetros, y otro externo, que nunca participó en la elección, mide.

**Diapositiva 10. Resultados de la base mundial.**
Mostrar: `fig_resumen_comparativa.png` (fila de arriba) y `fig_benchmark_roc_confusion.png`.
Qué decir: el SVM con kernel RBF es el mejor clasificador (F1 macro 0.82, AUC 0.96) y el SVR RBF el mejor en regresión (R² 0.81). KNN va detrás. Los lineales
quedan en F1 0.60 y R² 0.48. La referencia, que siempre predice la clase más común, da F1 0.29.

**Diapositiva 11. Pero, ¿qué aprendieron?**
Mostrar: `fig_benchmark_modelos.png`.
Qué decir:
- Al quitar latitud y longitud, KNN baja de 0.79 a 0.48 en R². Aprende dónde está la planta.
- Con partición aleatoria los resultados salen más altos que con bloques.
- Dejando un continente entero fuera, todos los modelos dan R² negativo, hasta la referencia.

El modelo no sirve fuera de las regiones que vio, y eso confirma el hallazgo del EDA.

**Diapositiva 12. Por qué los lineales quedan bajos.**
Mostrar: la interpretación de R11.
Qué decir: un modelo lineal ajusta un solo plano para todo el mundo, y el índice tiene un nivel distinto en cada región. A eso se le llama no estacionariedad
espacial (Brunsdon et al., 1996). KNN y SVM con kernel ajustan vecindarios locales. Para mejorar se podrían usar bosques aleatorios o XGBoost, modelos espaciales o
stacking. Pero subir el puntaje no hace que el índice mida aptitud.

**Diapositiva 13. La segunda línea: generación real en Colombia.**
Mostrar: `fig_pipeline_clima.png`.
Qué decir: para volver a la idea original del clima, construimos una base propia:
- Generación real hora por hora de 16 plantas colombianas, desde la API pública de XM.
- Clima de cada sitio desde Open-Meteo.
- Coordenadas verificadas con prensa y con las fichas de Global Energy Monitor.

El resultado son 8,589 días planta, de enero de 2024 a febrero de 2026, sin valores vacíos.

**Diapositiva 14. EDA de la base de Colombia.**
Mostrar: `fig_eda_base_colombia.png` y `fig_fc_vs_radiacion.png`.
Qué decir: lo que predecimos es el factor de capacidad, la energía del día dividida entre la máxima posible. El 84 % de su variación es de un día a otro dentro de
cada planta. La radiación es la variable que más se relaciona con él, y nubosidad, temperatura y humedad están muy correlacionadas con ella. Falta la tecnología
de cada planta (si tiene seguidor solar), que es pendiente.

**Diapositiva 15. Resultados en Colombia.**
Mostrar: `fig_resumen_comparativa.png` (fila de abajo) y `fig_benchmark_colombia.png`.
Qué decir:
- Aquí ganan los modelos lineales. Lasso explica R² de 0.47 del sube y baja diario, y el error baja del 20 % al 14 % del factor de capacidad medio.
- Clasificar el día como bajo, medio o alto: la regresión logística acierta 57 de cada 100, con AUC 0.77. El azar daría 33.
- Cada kWh/m² de radiación suma unos 4.7 puntos de factor de capacidad, como predice la física.
- La partición aleatoria vuelve a inflar los resultados.

**Diapositiva 16. Conclusiones.**
Mostrar: R12.
Qué decir: el índice del dataset no mide aptitud desde el clima y no se transfiere entre regiones. Con datos reales, el clima sí explica cerca de la mitad de la
variación diaria de la generación, y casi todo lo aporta la radiación.

**Diapositiva 17. Limitaciones y siguientes pasos.**
Mostrar: R13.
Qué decir: falta la tecnología de cada planta, radiación medida en lugar de modelada, datos por hora, y los modelos de la entrega siguiente (árboles, XGBoost, balanceo con SMOTE).

Preguntas probables y respuesta corta:
- ¿Por qué no partición aleatoria? Porque plantas vecinas se parecen y el resultado sale optimista. Lo mostramos con números en la diapositiva 11.
- ¿Por qué F1 macro y no exactitud? Porque con 75 % de clase Alta, predecir siempre Alta da 75 % de exactitud y no sirve para nada.
- ¿Por qué tan pocas plantas en Colombia? Porque XM no publica coordenadas, y solo usamos plantas cuya ubicación pudimos verificar con fuentes.
- ¿Qué es el AUC? La probabilidad de que el modelo le dé más puntaje a un ejemplo de la clase correcta que a uno de otra clase. 0.5 es azar y 1 es perfecto.

## R1. La pregunta y por qué cambió

La propuesta inicial era predecir la aptitud solar de un sitio a partir del clima, para tener algo útil en cualquier lugar. El dataset del proyecto
(Mantilla-Guerra et al., 2026) trae un índice de aptitud solar (`solar_aptitude`) calculado con la ecuación 1 del paper:
`IAS = 0.40 · pendiente + 0.25 · orientación + 0.20 · sombreado + 0.15 · curvatura`. El clima no entra en esa fórmula, así que no puede predecirse desde el clima.
El EDA confirmó además que el índice depende sobre todo de la región y del lote de procesamiento.

Por eso el trabajo tiene dos líneas:
1. Modelar el índice del dataset y mostrar, con validación espacial, por qué no se transfiere entre regiones.
2. Construir una base propia con la generación real de plantas colombianas y medir cuánto la explica el clima, que es la idea original.

## R2. Recolección de datos

| Fuente | Qué aporta | Acceso |
|---|---|---|
| Dataset de Mantilla-Guerra et al. (2026) | 58,978 plantas FV del mundo, 29 columnas: ubicación, terreno, clima anual, distancia a carretera, potencial FV y el índice | CSV del repositorio de los autores |
| Global Energy Monitor, Global Solar Power Tracker (release de febrero de 2026) | Base de plantas del dataset y fichas públicas con coordenadas | Web de GEM |
| XM, operador del mercado eléctrico colombiano | Generación horaria por planta y capacidad efectiva neta diaria, enero de 2024 a febrero de 2026 | API pública, sin clave |
| Open-Meteo, API histórica (Zippenfenig, 2023) | Radiación, nubosidad, temperatura, humedad y viento por hora, con los productos ERA5 (Hersbach et al., 2020) y `best_match` | API pública, sin clave |

![Pipeline del EDA](fig_pipeline_eda_mundial.png)

![Pipeline de la base de Colombia](fig_pipeline_clima.png)

## R3. EDA y limpieza de la base mundial

Problemas encontrados y cómo se resolvieron:

| Problema | Cifra | Decisión |
|---|---|---|
| Filas sin cobertura de terreno (DEM) | 64 filas, todas en clase Baja (3.4 % de esa clase) | Eliminar |
| Filas sin datos de clima o potencial | 3 filas, más 2 con viento en 0 | Eliminar |
| Duplicados de atributos | 1,598 filas iguales salvo el identificador | Eliminar |
| Orientación con valor -1 en terreno plano | 1,105 filas | Seno y coseno igual a 0 más un indicador `is_flat` |
| Distancia a carretera imposible | 65 filas a más de 100 km, máximo 2,296 km | Pasar a vacío y usar logaritmo |
| Área de la planta | 16,329 vacíos (27.7 %), densidades físicamente imposibles | Excluir |
| Variables que son agrupaciones de otras (`*_type`, `size`, `dt_wind`) | Rangos sin solapamiento | Excluir |

Resultado: 57,976 plantas y 15 predictoras (`dataset_eda_corregido.csv`, 57,976 × 22).

Decisiones y su respaldo:
- Los ángulos (orientación y dirección del viento) se pasan a seno y coseno, porque 359° y 1° son vecinos (Fisher, 1993).
- Se usa correlación de Spearman en lugar de Pearson por las colas pesadas y relaciones no lineales (Hollander et al., 2015).
- Los umbrales de VIF se interpretan con cautela (O'Brien, 2007).
- Se excluyen variables posteriores a la decisión de invertir (capacidad, estado operativo) para no meter información del futuro en el modelo (Kaufman et al., 2012).

Hallazgo central:
- El índice casi no se relaciona con las variables de terreno con las que se calcula: Spearman de -0.066 con la pendiente. Sí se relaciona con la longitud (+0.583).
- Un modelo con solo pendiente, orientación y curvatura explica R² de 0.18. Al añadir latitud y longitud sube a 0.91.
- Dejando fuera una macrorregión, el R² es negativo (-7.3 en Asia y Oceanía, -2.3 en América, -0.1 en Europa, África y Medio Oriente).
- El 94.5 % de la clase Baja viene de un lote europeo que se añadió en la segunda versión del dataset.

La consecuencia es que la validación debe ser espacial. Con partición aleatoria, plantas vecinas quedan a los dos lados y el resultado sale optimista
(Roberts et al., 2017; Ploton et al., 2020). Se definieron 496 bloques de 5° × 5°. En los 5 pliegues, la clase Baja queda entre 2.9 % y 3.8 %.

## R4. Construcción de la base de Colombia

1. Se listaron los 2,213 recursos solares de XM y se quedaron los 27 con al menos 1 MW de capacidad.
2. Se calculó el factor de capacidad diario: `FC = kWh del día / (capacidad efectiva neta × 24 h)`, con la capacidad de ese mismo día.
3. Coordenadas: XM no las publica. Nueve plantas se emparejaron con el dataset del paper y siete con la ficha pública de GEM. Cada emparejamiento lleva
   su motivo y una fuente de prensa o del desarrollador. Por ejemplo, el parque Fundación está en Pivijay (Magdalena). Se descartaron Palmira II, porque
   su ubicación no se confirmó, y Shangri-La, porque sus coordenadas en GEM contradicen su propia ficha.
4. Clima por hora de Open-Meteo en cada coordenada, con dos productos (`era5` y `best_match`). Se resumió por día: radiación diaria, nubosidad media en
   horas de luz, temperatura, humedad y viento. Las horas de XM y de Open-Meteo coinciden sin desfase, y el resumen diario evita problemas de alineación.
5. Filtros con criterios que no miran la variable que se predice: se excluyen los primeros 60 días de las plantas con arranque observado y las plantas con menos de 100 días útiles.

Resultado: `tabla_diaria_colombia.csv`, con 8,589 filas planta-día, 23 columnas, 16 plantas, 6 zonas de cercanía y ningún valor vacío fuera de `fc_ayer`.
Los días sin generación en XM van de 0 a 24 por planta.

EDA de la base:
- El 84 % de la varianza del FC es de un día a otro dentro de cada planta. El 16 % es diferencia entre plantas.
- Dentro de la planta, la radiación diaria explica cerca de la mitad del sube y baja del FC. Cada kWh/m² suma unos 4.7 puntos de FC, casi la proporción física simple.
- Nubosidad, temperatura y humedad están muy correlacionadas con la radiación.
- Pendiente: la tecnología de cada planta (seguidor o fija, potencia DC frente a AC). Ninguna de las 16 la tiene confirmada.

![EDA de la base de Colombia](fig_eda_base_colombia.png)

![FC contra radiación por planta](fig_fc_vs_radiacion.png)

## R5. Diseño experimental común

La partición va siempre primero, y todo lo que se aprende de los datos se ajusta solo con el entrenamiento de cada partición (Kaufman et al., 2012):

| Decisión | Base mundial | Base de Colombia | Respaldo |
|---|---|---|---|
| Unidad de partición | Bloques espaciales de 5° × 5°, 5 pliegues | Zona de plantas a menos de 100 km, más división en el tiempo (pasado para entrenar, futuro para probar) | Roberts et al. (2017); Ploton et al. (2020); Meyer y Pebesma (2021) |
| Imputación y escalado | Dentro de un `Pipeline` de scikit-learn | Dentro de un `Pipeline` | Pedregosa et al. (2011); Kaufman et al. (2012) |
| Nivel de cada planta y cortes de clase | No aplica | Calculados solo con el pasado de cada planta | Kaufman et al. (2012) |
| hyperparameter tuning | Validación anidada: 5 outer folds y 3 internos, todos por bloques | Validación anidada: 6 zonas externas, y en el interno se deja fuera una zona de entrenamiento a la vez | Varma y Simon (2006); Cawley y Talbot (2010) |
| Búsqueda | Grillas pequeñas en escala logarítmica para C y alpha | Igual | Bergstra y Bengio (2012) |
| Probabilidades del SVM lineal | Calibradas con `CalibratedClassifierCV` dentro del entrenamiento | Igual | Niculescu-Mizil y Caruana (2005) |
| SVM con kernel en la base mundial | Submuestra de 8,000 filas por pliegue, porque su costo crece entre cuadrático y cúbico con las filas | No hace falta | Chang y Lin (2011) |

Por qué validación anidada: si se eligen los hiperparámetros con los mismos datos con los que se reporta el resultado, la estimación sale optimista
(Varma y Simon, 2006; Cawley y Talbot, 2010). El bucle interno elige y el externo solo mide.

## R6. Modelos y por qué cada uno

| Modelo | Tarea | Por qué se incluye | Referencia |
|---|---|---|---|
| Referencia (clase mayoritaria, media o nivel de la planta) | Ambas | Mide el piso: cualquier modelo útil debe superarlo | |
| Regresión logística (y versión balanceada) | Clasificación | Lineal, interpretable. La versión con `class_weight="balanced"` compensa la clase Baja | Hastie et al. (2009); He y Garcia (2009) |
| Naive Bayes gaussiano | Clasificación | Modelo probabilístico simple y rápido, línea base del curso | Hastie et al. (2009) |
| KNN | Ambas | No supone forma de la relación. Sensible a la escala, por eso se estandariza | Cover y Hart (1967) |
| SVM lineal y SVM RBF | Clasificación | Margen máximo. El kernel RBF captura relaciones no lineales | Cortes y Vapnik (1995); Chang y Lin (2011) |
| Ridge | Regresión | Lineal con penalización L2, estable con variables correlacionadas | Hoerl y Kennard (1970) |
| Lasso | Regresión | Penalización L1, puede anular variables que sobran | Tibshirani (1996) |
| SVR lineal y SVR RBF | Regresión | Versión de regresión del SVM | Smola y Schölkopf (2004) |
| HistGradientBoosting | Regresión (Colombia, preliminar) | Control flexible basado en árboles | Friedman (2001) |

## R7. Métricas y por qué

| Métrica | Qué mide | Por qué se usa | Referencia |
|---|---|---|---|
| F1 macro | Promedio del F1 de cada clase | Métrica principal en clasificación. Con la clase Baja en 3 %, la exactitud premiaría ignorarla | He y Garcia (2009); Sokolova y Lapalme (2009) |
| Exactitud y exactitud balanceada | Aciertos totales y promedio del recall por clase | La balanceada no depende de la proporción de clases | Sokolova y Lapalme (2009) |
| Precisión y recall macro | Cuánto se acierta al predecir una clase y cuánto se recupera de ella | Separan los dos tipos de error | Sokolova y Lapalme (2009) |
| F1 de la clase Baja | F1 solo de la minoritaria | Es la clase difícil y la más afectada por el lote europeo | He y Garcia (2009) |
| AUC ROC macro uno contra el resto | Qué tan bien ordenan las probabilidades a cada clase frente a las demás, sin umbral | Complementa al F1, que depende del umbral | Fawcett (2006); Hand y Till (2001) |
| Kappa ponderado cuadrático | Acuerdo que castiga más los errores entre clases lejanas | Las clases tienen orden: confundir Baja con Alta es peor que con Media | Cohen (1968) |
| R² | Proporción de la varianza explicada | Métrica principal en regresión. En Colombia se calcula dentro de cada planta | Hastie et al. (2009) |
| RMSE y MAE | Error en las unidades de la variable. RMSE castiga más los errores grandes | Dan la magnitud del error | Hastie et al. (2009) |

## R8. Resultados de la base mundial

Con hiperparámetros por defecto, bloques de 5° y con latitud y longitud (clasificación):

| Modelo | F1 macro | Exactitud | Exactitud bal. | Precisión macro | Recall macro | F1 Baja | AUC | Kappa cuad. |
|---|---|---|---|---|---|---|---|---|
| Referencia | 0.285 | 0.748 | 0.333 | 0.249 | 0.333 | 0.000 | 0.500 | 0.000 |
| Regresión logística | 0.599 | 0.768 | 0.559 | 0.695 | 0.559 | 0.523 | 0.865 | 0.367 |
| Regresión logística balanceada | 0.629 | 0.719 | 0.798 | 0.587 | 0.798 | 0.507 | 0.873 | 0.481 |
| Naive Bayes | 0.448 | 0.574 | 0.657 | 0.515 | 0.657 | 0.169 | 0.768 | 0.261 |
| KNN | 0.787 | 0.897 | 0.742 | 0.859 | 0.742 | 0.658 | 0.951 | 0.737 |
| SVM lineal (calibrado) | 0.516 | 0.753 | 0.479 | 0.668 | 0.479 | 0.358 | 0.874 | 0.246 |
| SVM RBF | 0.812 | 0.896 | 0.772 | 0.869 | 0.772 | 0.747 | 0.958 | 0.751 |

Regresión, mismo escenario:

| Modelo | R² | RMSE | MAE |
|---|---|---|---|
| Referencia | -0.002 | 0.134 | 0.103 |
| Ridge | 0.479 | 0.096 | 0.069 |
| Lasso | 0.478 | 0.096 | 0.069 |
| KNN | 0.790 | 0.061 | 0.037 |
| SVR lineal | 0.479 | 0.096 | 0.069 |
| SVR RBF | 0.734 | 0.069 | 0.053 |

Los contrastes (F1 macro y R², por defecto):

| Modelo | F1 bloques con lat/lon | F1 bloques sin lat/lon | F1 aleatoria sin lat/lon | R² bloques con lat/lon | R² bloques sin lat/lon | R² aleatoria sin lat/lon |
|---|---|---|---|---|---|---|
| KNN | 0.787 | 0.658 | 0.730 | 0.790 | 0.480 | 0.656 |
| SVM / SVR RBF | 0.812 | 0.617 | 0.680 | 0.734 | 0.427 | 0.560 |
| Regresión logística / Ridge | 0.599 | 0.377 | 0.381 | 0.479 | 0.195 | 0.220 |

Dejando fuera una macrorregión completa, todos los modelos de regresión dan R² negativo, incluida la referencia.

Con hiperparámetros ajustados por validación anidada (5 outer folds y 3 internos, todos por bloques; bloques de 5° y con latitud y longitud):

| Modelo | F1 macro | Exactitud | Exactitud bal. | Precisión macro | Recall macro | F1 Baja | AUC | Kappa cuad. |
|---|---|---|---|---|---|---|---|---|
| Regresión logística | 0.600 | 0.768 | 0.560 | 0.694 | 0.560 | 0.525 | 0.865 | 0.368 |
| Regresión logística balanceada | 0.629 | 0.719 | 0.798 | 0.587 | 0.798 | 0.506 | 0.873 | 0.482 |
| Naive Bayes | 0.472 | 0.612 | 0.674 | 0.505 | 0.674 | 0.208 | 0.835 | 0.289 |
| KNN | 0.797 | 0.895 | 0.767 | 0.837 | 0.767 | 0.694 | 0.925 | 0.753 |
| SVM lineal (calibrado) | 0.516 | 0.753 | 0.479 | 0.668 | 0.479 | 0.358 | 0.874 | 0.246 |
| SVM RBF | 0.818 | 0.900 | 0.793 | 0.850 | 0.793 | 0.744 | 0.960 | 0.775 |

| Modelo | R² | RMSE | MAE |
|---|---|---|---|
| Ridge | 0.479 | 0.096 | 0.070 |
| Lasso | 0.479 | 0.096 | 0.069 |
| KNN | 0.796 | 0.060 | 0.036 |
| SVR lineal | 0.479 | 0.096 | 0.069 |
| SVR RBF | 0.806 | 0.059 | 0.037 |

El ajuste cambia poco a los modelos lineales y más a los no lineales:
- El SVR RBF sube de 0.734 a 0.806 en R². La búsqueda eligió un margen `epsilon` de 0.01 en lugar de 0.1, el valor por defecto. Con un índice que va de 0 a 0.94, un margen de 0.1 ignoraba errores grandes.
- KNN sube de 0.790 a 0.796 en R² y de 0.787 a 0.797 en F1, con 5 vecinos ponderados por distancia.
- Naive Bayes sube de 0.448 a 0.472 en F1 con más suavizado de la varianza.
- Los hiperparámetros elegidos son estables entre pliegues en KNN, Naive Bayes y SVR RBF, y en el SVM RBF el gamma fue `scale` en todos. En la regresión logística, Ridge y Lasso cambian entre pliegues sin que cambie el resultado, lo que indica una superficie plana: la regularización no es el cuello de botella.

![Benchmark mundial](fig_benchmark_modelos.png)

![ROC y matriz de confusión, base mundial](fig_benchmark_roc_confusion.png)

![Efecto del ajuste, base mundial](fig_benchmark_ajuste.png)

## R9. Resultados de la base de Colombia

Hiperparámetros ajustados con validación anidada (zona fuera y futuro de cada planta).

Regresión del FC diario:

| Modelo | R² dentro de planta (mediana) | Peor planta | Mejor planta | RMSE | MAE | MAE relativo |
|---|---|---|---|---|---|---|
| Referencia: nivel de la planta | -0.039 | -0.950 | 0.000 | 0.063 | 0.048 | 0.196 |
| Regresión lineal | 0.466 | -0.469 | 0.619 | 0.046 | 0.035 | 0.141 |
| Ridge | 0.461 | -0.473 | 0.613 | 0.046 | 0.035 | 0.141 |
| Lasso | 0.469 | -0.462 | 0.618 | 0.046 | 0.035 | 0.140 |
| KNN | 0.414 | -0.437 | 0.521 | 0.047 | 0.036 | 0.147 |
| SVR RBF | 0.407 | -0.292 | 0.528 | 0.047 | 0.035 | 0.144 |

Clasificación del día en bajo, medio o alto:

| Modelo | F1 macro | Exactitud | Exactitud bal. | Precisión macro | Recall macro | AUC | Kappa cuad. |
|---|---|---|---|---|---|---|---|
| Referencia | 0.178 | 0.364 | 0.333 | 0.121 | 0.333 | 0.500 | 0.000 |
| Regresión logística | 0.572 | 0.576 | 0.574 | 0.570 | 0.574 | 0.765 | 0.575 |
| Naive Bayes | 0.535 | 0.544 | 0.540 | 0.538 | 0.540 | 0.737 | 0.508 |
| KNN | 0.564 | 0.571 | 0.566 | 0.567 | 0.566 | 0.753 | 0.561 |
| SVM lineal (calibrado) | 0.542 | 0.558 | 0.559 | 0.539 | 0.559 | 0.751 | 0.553 |
| SVM RBF | 0.572 | 0.576 | 0.570 | 0.580 | 0.570 | 0.759 | 0.568 |

Hiperparámetros elegidos: la búsqueda tomó siempre la opción más suave (KNN con 101 vecinos, SVR con C de 0.1, Ridge con alpha de 100). Con 6 zonas, lo que generaliza
es suavizar mucho. KNN y SVR quedaron en el borde de su grilla.

Contrastes (por defecto): con solo la radiación, el SVR sube de 0.358 a 0.488 en R². Con partición aleatoria, el SVR sube de 0.358 a 0.479 y el SVM RBF de 0.570 a 0.598 en F1.
Las demás variables de clima agregan ruido, y la partición aleatoria infla el resultado.

Prueba de fuentes de clima (cuaderno de clima): la radiación de `best_match` explica más que la de `era5` en 13 de las 16 plantas (mediana de R² de 0.49 frente a 0.41).
Parte del límite del resultado es la calidad de la radiación modelada.

![Benchmark de Colombia](fig_benchmark_colombia.png)

![ROC y matriz de confusión, Colombia](fig_benchmark_colombia_roc_confusion.png)

## R10. Tabla comparativa de todos los modelos

Todos con hiperparámetros ajustados por validación anidada. La mejor fila de cada tarea va primero.

Base mundial, clasificación de la clase de aptitud (bloques de 5°, con latitud y longitud):

| Modelo | F1 macro | Exactitud bal. | AUC | Kappa cuad. | F1 Baja |
|---|---|---|---|---|---|
| SVM RBF | 0.818 | 0.793 | 0.960 | 0.775 | 0.744 |
| KNN | 0.797 | 0.767 | 0.925 | 0.753 | 0.694 |
| Regresión logística balanceada | 0.629 | 0.798 | 0.873 | 0.482 | 0.506 |
| Regresión logística | 0.600 | 0.560 | 0.865 | 0.368 | 0.525 |
| SVM lineal (calibrado) | 0.516 | 0.479 | 0.874 | 0.246 | 0.358 |
| Naive Bayes | 0.472 | 0.674 | 0.835 | 0.289 | 0.208 |
| Referencia | 0.285 | 0.333 | 0.500 | 0.000 | 0.000 |

Base mundial, regresión del índice:

| Modelo | R² | RMSE | MAE |
|---|---|---|---|
| SVR RBF | 0.806 | 0.059 | 0.037 |
| KNN | 0.796 | 0.060 | 0.036 |
| Ridge | 0.479 | 0.096 | 0.070 |
| SVR lineal | 0.479 | 0.096 | 0.069 |
| Lasso | 0.479 | 0.096 | 0.069 |
| Referencia | -0.002 | 0.134 | 0.103 |

Colombia, regresión del factor de capacidad diario (zona fuera y futuro de cada planta):

| Modelo | R² dentro de planta (mediana) | RMSE | MAE relativo |
|---|---|---|---|
| Lasso | 0.469 | 0.046 | 0.140 |
| Regresión lineal | 0.466 | 0.046 | 0.141 |
| Ridge | 0.461 | 0.046 | 0.141 |
| KNN | 0.414 | 0.047 | 0.147 |
| SVR RBF | 0.407 | 0.047 | 0.144 |
| Referencia | -0.039 | 0.063 | 0.196 |

Colombia, clasificación del día (bajo, medio, alto):

| Modelo | F1 macro | Exactitud bal. | AUC | Kappa cuad. |
|---|---|---|---|---|
| Regresión logística | 0.572 | 0.574 | 0.765 | 0.575 |
| SVM RBF | 0.572 | 0.570 | 0.759 | 0.568 |
| KNN | 0.564 | 0.566 | 0.753 | 0.561 |
| SVM lineal (calibrado) | 0.542 | 0.559 | 0.751 | 0.553 |
| Naive Bayes | 0.535 | 0.540 | 0.737 | 0.508 |
| Referencia | 0.178 | 0.333 | 0.500 | 0.000 |

Lo que cambia entre bases: en la mundial ganan los modelos locales (SVM RBF, SVR RBF, KNN), porque el índice depende de la región. En Colombia ganan los lineales, porque la energía es casi proporcional a la radiación y hay pocas zonas para aprender algo más complejo.

![Comparativa de todos los modelos](fig_resumen_comparativa.png)

## R11. Interpretación: por qué unos modelos quedan bajos y cómo se podría mejorar

### Base mundial

Ridge, Lasso, la regresión logística y el SVM lineal quedan muy por debajo de KNN y del SVM con kernel (R² de 0.48 frente a 0.79; F1 de 0.60 frente a 0.81). La razón es la forma del problema, no la falta de ajuste:

- Un modelo lineal ajusta un único plano para todo el mundo. El índice tiene un nivel distinto en cada región y la relación con el terreno cambia de una región a otra. A eso se le llama no estacionariedad espacial, y un modelo global lineal no puede representarla (Brunsdon et al., 1996).
- KNN y el SVM con kernel ajustan vecindarios locales. Con latitud y longitud entre las variables, aprenden el nivel de cada región. Por eso son los que más caen al quitar latitud y longitud (KNN pasa de 0.79 a 0.48 en R²).
- Lasso no ayuda porque el problema no es que sobren variables. Con 15 variables y 58 mil filas la regularización casi no cambia nada. Con el ajuste, Lasso eligió valores de alpha muy pequeños (de 0.00001 a 0.001), es decir, casi sin penalización, y su R² quedó igual (0.479).
- Naive Bayes es el más bajo porque supone que las variables son independientes dentro de cada clase y con distribución normal. Aquí están correlacionadas (temperatura, humedad, radiación y latitud) y varias no son normales (Hastie et al., 2009).

Qué se podría hacer, según la literatura:
1. Ensambles de árboles: bosques aleatorios (Breiman, 2001) y XGBoost (Chen y Guestrin, 2016). Capturan interacciones y no linealidades sin que haya que especificarlas, y los pide la entrega siguiente. Lo esperable es un resultado parecido al de KNN y SVM con kernel dentro de las regiones conocidas.
2. Modelos espaciales: un bosque aleatorio con distancias o coordenadas como covariables (Hengl et al., 2018), o una regresión geográficamente ponderada, que estima coeficientes distintos en cada lugar y hace explícita la no estacionariedad (Brunsdon et al., 1996).
3. Stacking: combinar las predicciones de varios modelos (lineal, KNN y SVM) con un modelo de segundo nivel entrenado con predicciones fuera de pliegue (Wolpert, 1992).
4. Una advertencia: subir el puntaje dentro de las regiones conocidas no arregla la transferencia entre continentes ni hace que el índice mida aptitud. El área de aplicabilidad (Meyer y Pebesma, 2021) sirve para delimitar dónde una predicción es confiable. Quitar el efecto del lote de procesamiento exigiría recalcular el índice con datos homogéneos, no un mejor modelo.

### Base de Colombia

Aquí pasa lo contrario: los lineales son los mejores (R² de 0.47) y los flexibles quedan por debajo (0.41). La física explica por qué. La energía de una planta es casi proporcional a la radiación que recibe, así que un modelo lineal ya tiene la forma correcta. Con solo 6 zonas, los modelos flexibles aprenden detalles de esas zonas que no se repiten en otras. Por eso la búsqueda de hiperparámetros eligió siempre la opción más suave.

El techo de alrededor de 0.5 viene de los datos, no del modelo:
- La radiación es modelada, no medida en la planta. Cambiar de producto (`era5` a `best_match`) subió la mediana de 0.41 a 0.49. Urraca et al. (2018) evaluaron la radiación de ERA5 contra estaciones y productos satelitales. Su lectura completa está pendiente.
- Los promedios diarios esconden la variación de las nubes dentro del día.
- El nivel de cada planta depende de su tecnología y de recortes o mantenimiento que no medimos, y en algunas plantas cambia con el tiempo.

Qué se podría hacer, según la literatura:
1. Un modelo híbrido físico y estadístico: calcular la radiación en el plano de los paneles y la temperatura de celda con pvlib (Holmgren et al., 2018), con la tecnología real de cada planta, y usar el aprendizaje automático solo para corregir el residuo. Antonanzas et al. (2016) revisan los enfoques físicos, estadísticos e híbridos para pronóstico FV.
2. Mejor radiación: productos satelitales o mediciones en estaciones en lugar de reanálisis.
3. Modelos de efectos mixtos con un intercepto aleatorio por planta (Bates et al., 2015), o un bosque aleatorio de efectos mixtos (Hajjem et al., 2014). Manejan el nivel de cada planta y la dependencia entre días de la misma planta dentro del modelo, en lugar de centrar a mano.
4. Datos horarios y un índice de claridad, para separar el efecto de las nubes del de la geometría solar (Antonanzas et al., 2016).
5. Ensambles de árboles: con 6 zonas probablemente aportan poco, porque los modelos flexibles ya quedaron por debajo del lineal. Con más plantas y zonas podrían ayudar.
6. Completar la tecnología de cada planta, que es lo que más probablemente explica el 16 % de la varianza entre plantas.

## R12. Conclusiones

1. El índice de aptitud del dataset no sirve para predecir aptitud desde el clima. Su fórmula solo usa terreno, y en la práctica depende sobre todo de la región y del lote de procesamiento.
2. Los modelos base del curso predicen bien el índice dentro de las regiones conocidas. El SVM RBF llega a F1 macro de 0.82 y AUC de 0.96, y el SVR RBF a R² de 0.81. Pero lo logran en buena parte reconociendo la ubicación: sin latitud y longitud caen, y dejando una región fuera todos dan R² negativo.
3. Con la validación espacial por bloques los resultados salen más bajos que con partición aleatoria. Esa diferencia es la que justifica la validación espacial.
4. La base de Colombia, construida con generación real de XM y clima de Open-Meteo para 16 plantas, muestra que el clima del día explica cerca de la mitad del sube y baja diario de una planta (R² de 0.47 a 0.49 dentro de planta). Casi todo lo aporta la radiación, y cada kWh/m² suma unos 4.7 puntos de factor de capacidad, como predice la física.
5. En Colombia los modelos lineales son los mejores. El límite está en los datos: radiación modelada, promedios diarios, pocas zonas y la tecnología de cada planta sin medir.
6. El hyperparameter tuning con validación anidada mejora poco los modelos lineales y algo más los no lineales (el SVR RBF mundial de 0.73 a 0.81). En casi todos los casos el cuello de botella son los datos y no los hiperparámetros.

## R13. Limitaciones y pendientes

- La tecnología de cada planta colombiana (seguidor o fija, potencia DC frente a AC) no se ha investigado. Es lo que más probablemente explica el 16 % de la varianza entre plantas.
- El clima de Colombia es modelado, no medido. Falta comparar con radiación medida por estaciones del IDEAM.
- Las grillas de hiperparámetros son pequeñas. En Colombia, KNN y SVR quedaron en el borde de su grilla.
- No se usaron todavía árboles de decisión, bosques aleatorios ni XGBoost, ni balanceo con SMOTE o ADASYN, que pide la entrega siguiente.
- Los SVM con kernel de la base mundial se entrenaron con una submuestra de 8,000 filas.
- Las referencias se verificaron contra Crossref, DataCite o la página de la revista. Antes de citarlas en el informe final hay que leer los textos completos.

## R14. Referencias

- Antonanzas, J., Osorio, N., Escobar, R., Urraca, R. et al. (2016). Review of photovoltaic power forecasting. *Solar Energy, 136*, 78-111. https://doi.org/10.1016/j.solener.2016.06.069
- Bates, D., Mächler, M., Bolker, B. y Walker, S. (2015). Fitting linear mixed-effects models using lme4. *Journal of Statistical Software, 67*(1). https://doi.org/10.18637/jss.v067.i01
- Bergstra, J. y Bengio, Y. (2012). Random search for hyper-parameter optimization. *Journal of Machine Learning Research, 13*, 281-305. https://jmlr.org/papers/v13/bergstra12a.html
- Breiman, L. (2001). Random forests. *Machine Learning, 45*(1), 5-32. https://doi.org/10.1023/A:1010933404324
- Brunsdon, C., Fotheringham, A. S. y Charlton, M. E. (1996). Geographically weighted regression: a method for exploring spatial nonstationarity. *Geographical Analysis, 28*(4), 281-298. https://doi.org/10.1111/j.1538-4632.1996.tb00936.x
- Cawley, G. C. y Talbot, N. L. C. (2010). On over-fitting in model selection and subsequent selection bias in performance evaluation. *Journal of Machine Learning Research, 11*, 2079-2107. https://jmlr.org/papers/v11/cawley10a.html
- Chang, C.-C. y Lin, C.-J. (2011). LIBSVM: a library for support vector machines. *ACM Transactions on Intelligent Systems and Technology, 2*, 1-27. https://doi.org/10.1145/1961189.1961199
- Chen, T. y Guestrin, C. (2016). XGBoost: a scalable tree boosting system. *Proceedings of the 22nd ACM SIGKDD International Conference on Knowledge Discovery and Data Mining*, 785-794. https://doi.org/10.1145/2939672.2939785
- Cohen, J. (1968). Weighted kappa: nominal scale agreement with provision for scaled disagreement or partial credit. *Psychological Bulletin, 70*, 213-220. https://doi.org/10.1037/h0026256
- Cortes, C. y Vapnik, V. (1995). Support-vector networks. *Machine Learning, 20*, 273-297. https://doi.org/10.1007/BF00994018
- Cover, T. y Hart, P. (1967). Nearest neighbor pattern classification. *IEEE Transactions on Information Theory, 13*, 21-27. https://doi.org/10.1109/TIT.1967.1053964
- Fawcett, T. (2006). An introduction to ROC analysis. *Pattern Recognition Letters, 27*, 861-874. https://doi.org/10.1016/j.patrec.2005.10.010
- Fisher, N. I. (1993). *Statistical analysis of circular data*. Cambridge University Press. https://doi.org/10.1017/CBO9780511564345
- Friedman, J. H. (2001). Greedy function approximation: a gradient boosting machine. *The Annals of Statistics*. https://doi.org/10.1214/aos/1013203451
- Global Energy Monitor. (2026). *Global Solar Power Tracker, February 2026 release*. https://globalenergymonitor.org/projects/global-solar-power-tracker
- Hajjem, A., Bellavance, F. y Larocque, D. (2014). Mixed-effects random forest for clustered data. *Journal of Statistical Computation and Simulation, 84*(6), 1313-1328. https://doi.org/10.1080/00949655.2012.741599
- Hand, D. J. y Till, R. J. (2001). A simple generalisation of the area under the ROC curve for multiple class classification problems. *Machine Learning, 45*, 171-186. https://doi.org/10.1023/A:1010920819831
- Hastie, T., Tibshirani, R. y Friedman, J. (2009). *The elements of statistical learning* (2.ª ed.). Springer. https://doi.org/10.1007/978-0-387-84858-7
- He, H. y Garcia, E. A. (2009). Learning from imbalanced data. *IEEE Transactions on Knowledge and Data Engineering, 21*, 1263-1284. https://doi.org/10.1109/TKDE.2008.239
- Hengl, T., Nussbaum, M., Wright, M. N., Heuvelink, G. B. M. et al. (2018). Random forest as a generic framework for predictive modeling of spatial and spatio-temporal variables. *PeerJ, 6*, e5518. https://doi.org/10.7717/peerj.5518
- Hersbach, H. et al. (2020). The ERA5 global reanalysis. *Quarterly Journal of the Royal Meteorological Society, 146*(730), 1999-2049. https://doi.org/10.1002/qj.3803
- Hoerl, A. E. y Kennard, R. W. (1970). Ridge regression: biased estimation for nonorthogonal problems. *Technometrics, 12*, 55-67. https://doi.org/10.1080/00401706.1970.10488634
- Hollander, M., Wolfe, D. A. y Chicken, E. (2015). *Nonparametric statistical methods*. Wiley. https://doi.org/10.1002/9781119196037
- Holmgren, W. F., Hansen, C. W. y Mikofski, M. A. (2018). pvlib python: a python package for modeling solar energy systems. *Journal of Open Source Software, 3*(29), 884. https://doi.org/10.21105/joss.00884
- Kaufman, S., Rosset, S., Perlich, C. y Stitelman, O. (2012). Leakage in data mining: formulation, detection, and avoidance. *ACM Transactions on Knowledge Discovery from Data, 6*(4), 1-21. https://doi.org/10.1145/2382577.2382579
- Mantilla-Guerra, A., Mejia-Escobar, C., Azorin-Lopez, J. y Garcia-Rodriguez, J. (2026). Global dataset of solar power plants: multidimensional integration and analysis. *Eng, 7*(7), 343. https://doi.org/10.3390/eng7070343
- Meyer, H. y Pebesma, E. (2021). Predicting into unknown space? Estimating the area of applicability of spatial prediction models. *Methods in Ecology and Evolution, 12*(9), 1620-1633. https://doi.org/10.1111/2041-210X.13650
- Niculescu-Mizil, A. y Caruana, R. (2005). Predicting good probabilities with supervised learning. *Proceedings of the 22nd International Conference on Machine Learning*, 625-632. https://doi.org/10.1145/1102351.1102430
- O'Brien, R. M. (2007). A caution regarding rules of thumb for variance inflation factors. *Quality and Quantity, 41*, 673-690. https://doi.org/10.1007/s11135-006-9018-6
- Pedregosa, F. et al. (2011). Scikit-learn: machine learning in Python. *Journal of Machine Learning Research, 12*, 2825-2830. https://jmlr.org/papers/v12/pedregosa11a.html
- Ploton, P. et al. (2020). Spatial validation reveals poor predictive performance of large-scale ecological mapping models. *Nature Communications, 11*, 4540. https://doi.org/10.1038/s41467-020-18321-y
- Roberts, D. R. et al. (2017). Cross-validation strategies for data with temporal, spatial, hierarchical, or phylogenetic structure. *Ecography, 40*(8), 913-929. https://doi.org/10.1111/ecog.02881
- Smola, A. J. y Schölkopf, B. (2004). A tutorial on support vector regression. *Statistics and Computing, 14*, 199-222. https://doi.org/10.1023/B:STCO.0000035301.49549.88
- Sokolova, M. y Lapalme, G. (2009). A systematic analysis of performance measures for classification tasks. *Information Processing and Management, 45*, 427-437. https://doi.org/10.1016/j.ipm.2009.03.002
- Tibshirani, R. (1996). Regression shrinkage and selection via the lasso. *Journal of the Royal Statistical Society B, 58*, 267-288. https://doi.org/10.1111/j.2517-6161.1996.tb02080.x
- Urraca, R. et al. (2018). Evaluation of global horizontal irradiance estimates from ERA5 and COSMO-REA6 reanalyses using ground and satellite-based data. *Solar Energy, 164*, 339-354. https://doi.org/10.1016/j.solener.2018.02.059
- Varma, S. y Simon, R. (2006). Bias in error estimation when using cross-validation for model selection. *BMC Bioinformatics, 7*, 91. https://doi.org/10.1186/1471-2105-7-91
- Wolpert, D. H. (1992). Stacked generalization. *Neural Networks, 5*(2), 241-259. https://doi.org/10.1016/S0893-6080(05)80023-1
- XM S.A. E.S.P. (2026). *API pública de datos del mercado de energía mayorista y portal SiMEM*. https://www.simem.co/
- Zippenfenig, P. (2023). *Open-Meteo.com Weather API* [software]. Zenodo. https://doi.org/10.5281/zenodo.7970649

## R15. Terminología

| Término | Qué es | Para qué se usó en este proyecto |
|---|---|---|
| Índice de aptitud solar (IAS, `solar_aptitude`) | Índice de 0 a 1 que el paper calcula con pendiente, orientación, sombreado y curvatura | Variable objetivo de la base mundial, en regresión y en sus tres clases |
| Factor de capacidad (FC) | Energía generada dividida entre la máxima posible a potencia nominal en el mismo tiempo | Variable objetivo de la base de Colombia |
| Capacidad efectiva neta | Potencia máxima que la planta puede entregar a la red, según XM | Denominador del factor de capacidad |
| Potencia DC y AC | Potencia pico de los paneles (DC) y la que sale del inversor (AC) | Su relación es parte de la tecnología pendiente de cada planta |
| Seguidor solar | Estructura que gira los paneles para seguir al sol | Tecnología pendiente; cambia el nivel de FC de cada planta |
| Radiación global horizontal (GHI) | Energía solar que llega a una superficie horizontal, en kWh/m² por día | Principal variable de clima en la base de Colombia |
| Nubosidad | Fracción del cielo cubierta por nubes | Variable de clima; se probó si aporta además de la radiación |
| Reanálisis (ERA5) | Reconstrucción del clima pasado que combina modelo y observaciones | Fuente del clima; se comparó con `best_match` |
| `best_match` | Opción por defecto de Open-Meteo, que combina IFS HRES, ERA5 y ERA5-Land | Fuente de clima que mejor explicó la generación |
| NOCT | Temperatura de operación nominal de una celda, usada para estimarla desde la temperatura del aire | Línea base física con pérdida por temperatura |
| Dato centinela | Valor que en realidad significa "sin dato" (un 0 o un -1) | Se detectaron y se eliminaron o recodificaron en el EDA |
| Fuga de información (data leakage) | Que el modelo use, al entrenar, información que no tendría en la práctica, como los datos de prueba | Se evitó con la partición primero y todo el preprocesamiento dentro del `Pipeline` |
| `Pipeline` | Cadena de pasos de scikit-learn que se ajustan juntos solo con el entrenamiento | Imputación, escalado, calibración y modelo en un solo objeto |
| Imputación por mediana | Rellenar un vacío con la mediana del entrenamiento | 63 vacíos de distancia a carretera en la base mundial |
| Estandarización | Restar la media y dividir entre la desviación estándar | Necesaria para KNN, SVM y modelos regularizados |
| Validación cruzada | Repetir entrenamiento y prueba en varias particiones | Base de toda la evaluación |
| Validación espacial por bloques | Partir por zonas geográficas enteras en vez de filas al azar | Evitar que plantas vecinas queden en entrenamiento y prueba a la vez |
| Partición temporal | Entrenar con el pasado y probar con el futuro | Base de Colombia: el nivel de cada planta sale de su pasado |
| Validación anidada | Bucle interno que elige hiperparámetros y bucle externo que mide | hyperparameter tuning sin inflar el resultado |
| Hiperparámetro | Parámetro que no se aprende de los datos y se fija antes (C, alpha, k) | Se ajustaron con búsqueda en grilla |
| Búsqueda en grilla | Probar todas las combinaciones de una lista de valores | hyperparameter tuning |
| Regularización (L1, L2) | Penalización que limita los coeficientes del modelo | Ridge (L2), Lasso (L1) y el parámetro C de logística y SVM |
| Kernel RBF | Función que permite al SVM separar datos con fronteras no lineales | SVM y SVR con kernel |
| Calibración de probabilidades | Ajustar las salidas del modelo para que se comporten como probabilidades | Probabilidades del SVM lineal, necesarias para el AUC |
| Clase desbalanceada | Clase con muchas menos observaciones que las demás | La clase Baja (3 %) en la base mundial |
| `class_weight="balanced"` | Dar más peso en el entrenamiento a las clases poco frecuentes | Regresión logística balanceada |
| F1 macro | Promedio simple del F1 de cada clase | Métrica principal de clasificación |
| AUC ROC | Probabilidad de que el modelo ordene bien un ejemplo positivo frente a uno negativo | Medir la calidad de las probabilidades sin depender de un umbral |
| Kappa ponderado cuadrático | Acuerdo entre predicción y realidad que castiga más los errores entre clases lejanas | Clases con orden (Baja, Media, Alta; bajo, medio, alto) |
| R² | Proporción de la varianza explicada por el modelo | Métrica principal de regresión |
| RMSE y MAE | Raíz del error cuadrático medio y error absoluto medio | Magnitud del error en unidades de la variable |
| R² dentro de planta | R² calculado con la variable centrada en la media de cada planta | Medir cuánto explica el clima el sube y baja diario, sin el nivel de cada planta |
| Leave-region-out | Entrenar sin una región entera y probar en ella | Mostrar que el modelo del índice no se transfiere entre continentes |
| Bootstrap por planta | Remuestrear plantas, no filas, para obtener intervalos de confianza | Intervalos del R² en el cuaderno de clima |
| Prueba de Wilcoxon | Prueba no paramétrica para diferencias pareadas | Comparar modelos planta por planta en el cuaderno de clima |

## Referencias

- Mantilla-Guerra, A., Mejia-Escobar, C., Azorin-Lopez, J. y Garcia-Rodriguez, J. (2026). Global dataset of solar power plants: multidimensional integration and analysis. *Eng, 7*(7), 343. https://doi.org/10.3390/eng7070343
- Meyer, H. y Pebesma, E. (2021). Predicting into unknown space? Estimating the area of applicability of spatial prediction models. *Methods in Ecology and Evolution, 12*(9), 1620-1633. https://doi.org/10.1111/2041-210X.13650
- Ploton, P. et al. (2020). Spatial validation reveals poor predictive performance of large-scale ecological mapping models. *Nature Communications, 11*, 4540. https://doi.org/10.1038/s41467-020-18321-y
- Roberts, D. R. et al. (2017). Cross-validation strategies for data with temporal, spatial, hierarchical, or phylogenetic structure. *Ecography, 40*(8), 913-929. https://doi.org/10.1111/ecog.02881